# 11.08 - CLIP zero-shot image classification

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Offline CLIP-boundary classifier and prompt comparison.

Practice the exact embedding, prompt, and class-mapping mechanics of zero-shot CLIP with deterministic cached-like embeddings, avoiding a required checkpoint download.

## Core Ideas

CLIP embeds images and text into one normalized space. Zero-shot class logits come from scaled cosine similarities. Prompt wording changes text embeddings, so compare templates on identical images and labels. A package being allowlisted does not guarantee that a pretrained checkpoint is cached or competition-legal.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score

SEED = 11
np.random.seed(SEED)
torch.manual_seed(SEED)

## Prepared Cached-Like Encoder Outputs

Three classes use plain and descriptive prompt embeddings. Fifteen image embeddings and labels form the complete evaluation fixture; no network access is required.

In [ ]:
class_names = ["red square", "green circle", "blue triangle"]
prompt_templates = ["{}", "a centered photo of a {}"]
class_axes = torch.tensor([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,1.,0.]])
text_embeddings = torch.stack([class_axes + torch.tensor([0.00,0.00,0.00,0.04]), class_axes + torch.tensor([0.03,0.02,0.01,0.00])], dim=1)
image_labels = torch.arange(3).repeat_interleave(5)
image_embeddings = class_axes[image_labels] + 0.14 * torch.randn(15, 4)
print("images/text/support:", image_embeddings.shape, text_embeddings.shape, torch.bincount(image_labels).tolist())

## Exercise 11-A: Build prompt strings

Format every template for every class while preserving class-major order.

**Return structure — `build_class_prompts`:** A `list[list[str]]` of length `C`; every inner list has `P` prompt strings in template order.

In [ ]:
# TODO 11-A
def build_class_prompts(names, templates):
    raise NotImplementedError("Complete Exercise 11-A")


# Smoke check: inspect all class/template combinations.
class_prompts = build_class_prompts(class_names, prompt_templates)
print(class_prompts)

## Exercise 11-B: Create normalized class prototypes

Choose one prompt index or average all normalized prompts, then normalize again.

**Return structure — `make_class_prototypes`:** A CPU float32 tensor `[C,D]` with unit-norm rows. `prompt_index=None` means ensemble all `P` prompts.

In [ ]:
# TODO 11-B
def make_class_prototypes(embeddings, prompt_index=None):
    raise NotImplementedError("Complete Exercise 11-B")


# Smoke check: create plain and ensemble prototypes.
plain_prototypes = make_class_prototypes(text_embeddings, prompt_index=0)
ensemble_prototypes = make_class_prototypes(text_embeddings)
print(plain_prototypes.shape, ensemble_prototypes.shape)

## Exercise 11-C: Classify without training

Normalize images, calculate scaled cosine logits, and map argmax indices back to class IDs.

**Return structure — `zero_shot_classify`:** A dictionary with `probabilities` (CPU float32 `[N,C]`), `predictions` (CPU int64 `[N]`), and `confidence` (CPU float32 `[N]`).

In [ ]:
# TODO 11-C
def zero_shot_classify(images, prototypes, logit_scale=10.0):
    raise NotImplementedError("Complete Exercise 11-C")


# Smoke check: classify the full image fixture with both strategies.
plain_result = zero_shot_classify(image_embeddings, plain_prototypes)
ensemble_result = zero_shot_classify(image_embeddings, ensemble_prototypes)
print("ensemble predictions:", ensemble_result["predictions"].tolist())

## Exercise 11-D: Compare prompts on complete evidence

Report support, Macro-F1, mean confidence, and change from the first strategy.

**Return structure — `prompt_comparison`:** A two-row DataFrame with columns `strategy`, `sample_count`, `class_support`, `macro_f1`, `mean_confidence`, and `delta_macro_f1`.

In [ ]:
# TODO 11-D
def prompt_comparison(named_results, labels):
    raise NotImplementedError("Complete Exercise 11-D")


# Smoke check: print the complete prompt evidence.
prompt_table = prompt_comparison({"plain": plain_result, "ensemble": ensemble_result}, image_labels)
print(prompt_table.to_string(index=False))

## Test Cases

**Return structure — `run_day11_tests`:** Returns `None`; assertions and `Day 11 tests passed` communicate success.

In [ ]:
def run_day11_tests():
    assert len(class_prompts) == 3 and all(len(row) == 2 for row in class_prompts)
    assert plain_prototypes.shape == ensemble_prototypes.shape == (3, 4)
    assert torch.allclose(ensemble_prototypes.norm(dim=1), torch.ones(3), atol=1e-5)
    assert ensemble_result["probabilities"].shape == (15, 3)
    assert torch.allclose(ensemble_result["probabilities"].sum(dim=1), torch.ones(15), atol=1e-6)
    assert prompt_table.shape == (2, 6) and all(value == [5, 5, 5] for value in prompt_table["class_support"])
    print("Day 11 tests passed")


run_day11_tests()

## Day 11 Checklist

- [ ] Preserve class-major prompt ordering.
- [ ] Normalize before and after prompt averaging.
- [ ] Map argmax indices to the documented class order.
- [ ] Compare prompts on all labeled images.
- [ ] Run the test cases.